# KrepšelisAI — DI agentas, kuris randa pigiausią pirkinių krepšelį

**Generatyvinio DI sistemos prototipas, sukurtas su Google Gemini API**

| | |
|---|---|
| **Komanda** | Edvin Vilkanec, Gabriel Baranovskij |
| **Grupė** | PRIfs-24, VILNIUSTECH |
| **Modelis** | Google Gemini (Flash šeima) per `google-genai` SDK |
| **Repozitorija** | [github.com/gabrielbaranovskij/krepselisai](https://github.com/gabrielbaranovskij/krepselisai) |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabrielbaranovskij/krepselisai/blob/main/notebooks/KrepselisAI_Gemini_agentas.ipynb)

> **Vartotojas:** „Reikia pieno, duonos, 10 kiaušinių ir sviesto“
>
> **KrepšelisAI:** „Pigiausia apsipirkti Rimi: su kortele 7,66 €. Brangiausia IKI, 8,02 €, tad sutaupysite 0,36 €.“

### Turinys
1. Sistemos apžvalga
2. Aplinkos paruošimas
3. Duomenys: kainų katalogas ir etaloninės (X, y) poros
4. Agento įrankiai (deterministinis kodas)
5. DI agentas: Gemini + funkcijų kvietimas
6. Demonstracija: nuo įvesties iki išvesties, žingsnis po žingsnio
7. Rezultatų vizualizacija
8. Išbandykite patys
9. Vertinimas su etaloninėmis (X, y) poromis
10. Išvados ir apribojimai

## 1. Sistemos apžvalga

### Problema
Lietuvoje tas pats pirkinių krepšelis skirtinguose prekybos tinkluose (Rimi, IKI, Norfa) kainuoja skirtingai, o lojalumo kortelių nuolaidos skirtumą dar labiau padidina. Rankiniu būdu lyginti kiekvieną prekę trijose svetainėse užtrunka.

### Sprendimas
**KrepšelisAI** — pokalbių asistentas. Vartotojas laisvu tekstu parašo, ką nori nusipirkti, o sistema:

1. **supranta užklausą** — ištraukia prekes ir kiekius („pusė kilogramo sūrio“ → `suris_kg`, 0,5 kg);
2. **suderina prekes su katalogu** — skirtingų tinklų pavadinimai susiejami su viena preke;
3. **apskaičiuoja krepšelio kainą** kiekviename tinkle, su kortele ir be jos, ir randa geriausią 2 parduotuvių derinį;
4. **paaiškina rezultatą** žmogiška kalba.

### Kodėl tai agentas, o ne vienas užklausos–atsakymo kvietimas
Gemini modelis gauna **įrankius** (funkcijas) ir pats nusprendžia, kada ir su kokiais argumentais juos kviesti. Agento ciklas:

```
įvestis → Gemini planuoja → kviečia įrankį → gauna rezultatą → (kartoja) → galutinis atsakymas
```

### Kas daroma DI, o kas — įprastu kodu
| Užduotis | Kas atlieka | Kodėl |
|---|---|---|
| Laisvo teksto supratimas, kiekių ir vienetų atpažinimas | **Gemini** | Kalbos supratimas — LLM stiprybė |
| Prekių susiejimas su katalogu | **Gemini** (pagal katalogą iš įrankio) | Reikia semantinio palyginimo |
| Sumų, pigiausio tinklo, derinio skaičiavimas | **Python įrankis** | Skaičiai turi būti tikslūs, LLM jų neišgalvoja |
| Atsakymo suformulavimas | **Gemini** | Natūrali, draugiška kalba |

Toks padalijimas sumažina „haliucinacijų“ riziką: modelis niekada pats nesugalvoja kainų.

In [ ]:
# Sistemos architektūros schema (vizualizacija)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def deze(ax, x, y, w, h, tekstas, spalva, tekstospalva="#0b0b0b"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=spalva, ec="#52514e", lw=1))
    ax.text(x + w / 2, y + h / 2, tekstas, ha="center", va="center", fontsize=10, color=tekstospalva)

def rodykle(ax, a, b, tekstas=""):
    ax.add_patch(FancyArrowPatch(a, b, arrowstyle="-|>", mutation_scale=14, lw=1.4, color="#52514e"))
    if tekstas:
        ax.text((a[0] + b[0]) / 2, (a[1] + b[1]) / 2 + 0.12, tekstas, ha="center", fontsize=8.5, color="#52514e")

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.set_xlim(0, 12); ax.set_ylim(0, 4.2); ax.axis("off")
deze(ax, 0.2, 1.7, 1.8, 0.9, "Vartotojas\n(laisvas tekstas)", "#eef3fb")
deze(ax, 3.0, 1.5, 2.4, 1.3, "Gemini agentas\nplanuoja ir\nkviečia įrankius", "#2a78d6", "white")
deze(ax, 6.6, 2.9, 2.6, 0.9, "Įrankis 1\ngauti_kataloga()", "#f2f1ed")
deze(ax, 6.6, 0.4, 2.6, 0.9, "Įrankis 2\napskaiciuoti_krepseli()", "#f2f1ed")
deze(ax, 10.0, 2.9, 1.8, 0.9, "Kainų katalogas\nkainos.csv", "#fdf3e1")
deze(ax, 10.0, 0.4, 1.8, 0.9, "Rezultatas\nJSON + tekstas", "#e3f5ee")
rodykle(ax, (2.0, 2.15), (3.0, 2.15), "užklausa")
rodykle(ax, (5.4, 2.5), (6.6, 3.3), "function call")
rodykle(ax, (5.4, 1.8), (6.6, 0.9), "function call")
rodykle(ax, (9.2, 3.35), (10.0, 3.35))
rodykle(ax, (9.2, 0.85), (10.0, 0.85))
ax.set_title("KrepšelisAI agento architektūra", fontsize=13, loc="left", color="#0b0b0b")
plt.show()

## 2. Aplinkos paruošimas

Naudojamas oficialus Google **`google-genai`** SDK.

**API raktas.** Raktą galima gauti nemokamai [Google AI Studio](https://aistudio.google.com/apikey). Colab'e jį įrašykite kairėje esančiame **🔑 Secrets** skydelyje pavadinimu `GEMINI_API_KEY` ir įjunkite „Notebook access“. Raktas į kodą neįrašomas.

In [ ]:
%pip install -q -U google-genai pandas matplotlib

In [ ]:
import os, json, time, subprocess
from pathlib import Path
from dataclasses import dataclass, field
from itertools import combinations
from decimal import Decimal, ROUND_HALF_UP

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

from google import genai
from google.genai import types


def gauti_api_rakta() -> str:
    # 1) Colab Secrets, 2) aplinkos kintamasis, 3) įvedimas ranka
    try:
        from google.colab import userdata
        raktas = userdata.get("GEMINI_API_KEY")
        if raktas:
            return raktas
    except Exception:
        pass
    if os.environ.get("GEMINI_API_KEY"):
        return os.environ["GEMINI_API_KEY"]
    from getpass import getpass
    return getpass("Įveskite Gemini API raktą: ")


client = genai.Client(api_key=gauti_api_rakta())
print("Gemini klientas sukurtas, google-genai versija:", genai.__version__)

**Modelio pasirinkimas.** Google nuolat atnaujina modelius, todėl kodas bando kelis Flash šeimos modelius iš eilės ir naudoja pirmą, kuris pasiekiamas su jūsų raktu. Flash modeliai greiti, pigūs ir palaiko funkcijų kvietimą (*function calling*).

In [ ]:
MODELIU_KANDIDATAI = [
    "gemini-3.8-flash",
    "gemini-3.6-flash",
    "gemini-3-flash-preview",
    "gemini-2.5-flash",
]

def parinkti_modeli() -> str:
    for m in MODELIU_KANDIDATAI:
        try:
            client.models.get(model=m)
            return m
        except Exception as e:
            print(f"  {m}: nepasiekiamas ({type(e).__name__})")
    raise RuntimeError("Nė vienas modelis nepasiekiamas. Patikrinkite API raktą.")

MODEL = parinkti_modeli()
print("Naudojamas modelis:", MODEL)

## 3. Duomenys: kainų katalogas ir etaloninės (X, y) poros

Duomenys imami iš projekto repozitorijos:

| Failas | Turinys |
|---|---|
| `data/kainos.csv` | Pavyzdinis katalogas: 10 prekių × 3 tinklai, kaina be kortelės ir su kortele. Kiekvienas tinklas prekę vadina savaip (`tinklo_pavadinimas`). |
| `data/poros.jsonl` | 4 etaloninės poros: **X** — vartotojo užklausa, **y** — teisingas atsakymas. Jomis vertinsime agentą 9 skyriuje. |

> Kainos sugalvotos testavimui, jos nėra realios. Tikroje sistemoje katalogą kasdien pildytų scraperiai.

In [ ]:
REPO_URL = "https://github.com/gabrielbaranovskij/krepselisai.git"

def rasti_repo() -> Path:
    for kandidatas in [Path("."), Path(".."), Path("krepselisai")]:
        if (kandidatas / "data" / "kainos.csv").exists():
            return kandidatas.resolve()
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "krepselisai"], check=True)
    return Path("krepselisai").resolve()

ROOT = rasti_repo()
kainos_df = pd.read_csv(ROOT / "data" / "kainos.csv")
etalonai = [json.loads(e) for e in (ROOT / "data" / "poros.jsonl").read_text(encoding="utf-8").splitlines() if e.strip()]

print(f"Repozitorija: {ROOT}")
print(f"Katalogas: {kainos_df['preke_id'].nunique()} prekės × {kainos_df['tinklas'].nunique()} tinklai = {len(kainos_df)} eilučių")
print(f"Etaloninės poros: {len(etalonai)}")
kainos_df.head(6)

Kainos su lojalumo kortele. **Paryškinta** — pigiausias tinklas kiekvienai prekei. Matyti, kad nė vienas tinklas nėra pigiausias visoms prekėms, todėl vertinti reikia viso krepšelio.

In [ ]:
TINKLAI = ["Rimi", "IKI", "Norfa"]
lentele = (kainos_df.pivot_table(index="preke", columns="tinklas", values="kaina_su_kortele")[TINKLAI])
display(lentele.style.format("{:.2f} €").highlight_min(axis=1, props="font-weight:bold; background-color:#e3f5ee"))

pigiausias_kiek = lentele.idxmin(axis=1).value_counts().reindex(TINKLAI, fill_value=0)
print("Kiek prekių pigiausios kiekviename tinkle:", dict(pigiausias_kiek))

## 4. Agento įrankiai (deterministinis kodas)

Agentas turi du įrankius — paprastas Python funkcijas. Gemini mato tik jų **aprašus** (pavadinimą, paskirtį, parametrų schemą) ir pats sprendžia, kada juos kviesti.

| Įrankis | Ką daro | Kodėl reikalingas |
|---|---|---|
| `gauti_kataloga()` | Grąžina katalogo prekes: `preke_id`, pavadinimą, vienetą (`vnt` ar `kg`) ir kaip prekė vadinama kiekviename tinkle | Modelis turi žinoti, su kokiais ID susieti vartotojo prekes |
| `apskaiciuoti_krepseli(prekes, nerasta)` | Apskaičiuoja sumas kiekviename tinkle, pigiausią tinklą, sutaupymą ir geriausią 2 parduotuvių derinį | Tikslūs skaičiai be LLM klaidų |

Derinys laikomas vertu, jei jis bent **1,00 €** pigesnis už pigiausią vieną tinklą (mažiau neapsimoka važiuoti į antrą parduotuvę).

In [ ]:
# Katalogas atmintyje: (preke_id, tinklas) -> kainos
KAINOS = {(r.preke_id, r.tinklas): {"be_korteles": float(r.kaina), "su_kortele": float(r.kaina_su_kortele)}
          for r in kainos_df.itertuples()}
PREKES = {r.preke_id: {"preke": r.preke, "vienetas": r.vienetas} for r in kainos_df.itertuples()}
DERINIO_SLENKSTIS_EUR = 1.00


def D(x) -> Decimal:
    return Decimal(str(x))


def pinigai(x) -> float:
    # Pinigus apvaliname kaip buhalterijoje: Decimal, 0,005 -> 0,01 (ne float, kad nebūtų 13,254999… klaidų)
    return float(D(x).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


def gauti_kataloga() -> dict:
    # Įrankis 1: grąžina prekių sąrašą be kainų (kainas skaičiuoja tik įrankis 2)
    prekes = []
    for pid, p in PREKES.items():
        pavadinimai = kainos_df.loc[kainos_df.preke_id == pid, ["tinklas", "tinklo_pavadinimas"]]
        prekes.append({
            "preke_id": pid,
            "pavadinimas": p["preke"],
            "vienetas": p["vienetas"],
            "pavadinimai_tinkluose": dict(zip(pavadinimai.tinklas, pavadinimai.tinklo_pavadinimas)),
        })
    return {"prekes": prekes, "tinklai": TINKLAI}


def apskaiciuoti_krepseli(prekes: list, nerasta: list | None = None) -> dict:
    # Įrankis 2: visi skaičiavimai. prekes = [{"preke_id": ..., "kiekis": ...}, ...]
    nerasta = list(nerasta or [])
    sarasas = []
    for e in prekes:
        pid, kiekis = e.get("preke_id"), float(e.get("kiekis", 1) or 1)
        if pid not in PREKES:                     # nežinomas ID -> į „nerasta“, o ne klaida
            nerasta.append(str(pid))
            continue
        sarasas.append({"preke_id": pid, "preke": PREKES[pid]["preke"],
                        "kiekis": int(kiekis) if kiekis.is_integer() else round(kiekis, 3), "vienetas": PREKES[pid]["vienetas"]})
    if not sarasas:
        return {"klaida": "Nė viena prekė nerasta kataloge.", "nerasta": nerasta}

    def suma(tinklas, tipas):
        return pinigai(sum(D(KAINOS[(e["preke_id"], tinklas)][tipas]) * D(e["kiekis"]) for e in sarasas))

    sumos = {t: {"be_korteles": suma(t, "be_korteles"), "su_kortele": suma(t, "su_kortele")} for t in TINKLAI}
    pig_su = min(TINKLAI, key=lambda t: sumos[t]["su_kortele"])
    pig_be = min(TINKLAI, key=lambda t: sumos[t]["be_korteles"])
    brang_su = max(TINKLAI, key=lambda t: sumos[t]["su_kortele"])

    # Geriausias 2 parduotuvių derinys: kiekvieną prekę imame iš pigesnio tinklo
    geriausias = None
    for a, b in combinations(TINKLAI, 2):
        pirkti, viso = {a: [], b: []}, D(0)
        for e in sarasas:
            ka, kb = KAINOS[(e["preke_id"], a)]["su_kortele"], KAINOS[(e["preke_id"], b)]["su_kortele"]
            pirkti[a if ka <= kb else b].append(e["preke"])
            viso += D(min(ka, kb)) * D(e["kiekis"])
        viso = pinigai(viso)
        if geriausias is None or viso < geriausias["suma"]:
            geriausias = {"tinklai": [a, b], "suma": viso, "pirkti": pirkti}
    sutaupoma = pinigai(D(sumos[pig_su]["su_kortele"]) - D(geriausias["suma"]))
    geriausias["sutaupoma_lyginant_su_pigiausiu"] = sutaupoma
    geriausias["verta"] = sutaupoma >= DERINIO_SLENKSTIS_EUR

    return {
        "atpazintas_sarasas": sarasas,
        "nerasta": nerasta,
        "sumos_eur": sumos,
        "pigiausias_tinklas": {"su_kortele": pig_su, "be_korteles": pig_be},
        "brangiausias_tinklas_su_kortele": brang_su,
        "sutaupoma_vs_brangiausia_eur": pinigai(D(sumos[brang_su]["su_kortele"]) - D(sumos[pig_su]["su_kortele"])),
        "geriausias_2_parduotuviu_derinys": geriausias,
    }


IRANKIAI = {"gauti_kataloga": gauti_kataloga, "apskaiciuoti_krepseli": apskaiciuoti_krepseli}

**Įrankio patikra.** Prieš jungiant DI, įsitikiname, kad skaičiavimo įrankis atkuria etalonines sumas, kai jam paduodamas teisingai atpažintas sąrašas. Taip vėliau žinosime: jei agentas suklys, klaida bus DI atpažinimo žingsnyje, o ne skaičiavime.

> **Pastaba dėl apvalinimo.** Įrankis pinigus apvalina buhalteriškai (`Decimal`, 0,005 → 0,01). Etaloniniai failai buvo sugeneruoti su `float` apvalinimu, todėl ten, kur suma baigiasi lygiai pusę cento (pvz. 13,255 €), etalone yra 13,25 €, o įrankis duoda 13,26 €. Todėl sumas lyginame **±0,01 €** tikslumu.

In [ ]:
def artimi(a, b, tol=0.011) -> bool:
    # Lygina du objektus; skaičiai laikomi lygiais, jei skiriasi ne daugiau kaip 1 centu
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(artimi(a[k], b[k], tol) for k in a)
    if isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
        return abs(a - b) <= tol
    return a == b


for i, et in enumerate(etalonai, 1):
    y = et["y"]
    r = apskaiciuoti_krepseli(y["atpazintas_sarasas"], y["nerasta"])
    assert artimi(r["sumos_eur"], y["sumos_eur"]), f"Pavyzdys {i}: sumos nesutampa"
    assert r["pigiausias_tinklas"] == y["pigiausias_tinklas"]
    assert artimi(r["geriausias_2_parduotuviu_derinys"]["suma"], y["geriausias_2_parduotuviu_derinys"]["suma"])
    print(f"✓ Pavyzdys {i:02d}: sumos sutampa su etalonu ({r['pigiausias_tinklas']['su_kortele']}, "
          f"{r['sumos_eur'][r['pigiausias_tinklas']['su_kortele']]['su_kortele']:.2f} €)")

## 5. DI agentas: Gemini + funkcijų kvietimas

### 5.1 Sisteminė instrukcija
Instrukcija apibrėžia agento vaidmenį, darbo eigą ir kiekių atpažinimo taisykles (perimtos iš projekto `prompts/system_prompt.md`).

In [ ]:
SISTEMINE_INSTRUKCIJA = '''
Tu esi KrepšelisAI, pirkinių asistentas Lietuvos pirkėjams. Tavo tikslas: pagal vartotojo pirkinių sąrašą
rasti, kuriame prekybos tinkle (Rimi, IKI, Norfa) pigiausia nupirkti visą krepšelį.

Darbo eiga:
1. Iškviesk gauti_kataloga(), kad sužinotum, kokios prekės yra kataloge.
2. Iš vartotojo teksto ištrauk prekes ir kiekius, kiekvieną susiek su katalogo preke_id.
   - Prekė su vienetu "vnt" skaičiuojama pakuotėmis: "2 l pieno" = 2 (pienas 1 l), "20 kiaušinių" = 2 (pakuotė po 10).
   - Prekė su vienetu "kg" skaičiuojama kilogramais: "pusė kilogramo sūrio" = 0.5, "pusantro kilogramo" = 1.5.
   - Jei kiekis nenurodytas, imk 1 pakuotę arba 1 kg.
   - Prekes, kurių kataloge nėra, įrašyk į "nerasta" su kiekiu, pvz. "avokadai (3 vnt.)".
3. Iškviesk apskaiciuoti_krepseli() su atpažintu sąrašu. Pats kainų ir sumų neskaičiuok ir neišgalvok.
4. Pagal įrankio rezultatą parašyk atsakymą lietuviškai, 2–3 sakiniais, draugišku tonu:
   pigiausias tinklas su suma (su kortele), sutaupymas lyginant su brangiausiu, 2 parduotuvių derinys
   (tik jei verta = true) ir nerastos prekės. Sumas rašyk su kableliu, pvz. 7,66 €.
'''.strip()
print(SISTEMINE_INSTRUKCIJA)

### 5.2 Įrankių aprašai modeliui
Gemini įrankius „mato“ kaip JSON schemas. Kuo tikslesnis aprašas, tuo patikimiau modelis pateikia teisingus argumentus.

In [ ]:
IRANKIU_APRASAI = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="gauti_kataloga",
        description="Grąžina visų katalogo prekių sąrašą: preke_id, pavadinimą, vienetą (vnt arba kg) "
                    "ir kaip prekė vadinama kiekviename tinkle. Kviesk pirmiausia.",
        parameters_json_schema={"type": "object", "properties": {}},
    ),
    types.FunctionDeclaration(
        name="apskaiciuoti_krepseli",
        description="Apskaičiuoja krepšelio sumas Rimi, IKI ir Norfa tinkluose (su kortele ir be), "
                    "pigiausią tinklą, sutaupymą ir geriausią 2 parduotuvių derinį.",
        parameters_json_schema={
            "type": "object",
            "properties": {
                "prekes": {
                    "type": "array",
                    "description": "Atpažintos katalogo prekės.",
                    "items": {
                        "type": "object",
                        "properties": {
                            "preke_id": {"type": "string", "description": "ID iš gauti_kataloga()"},
                            "kiekis": {"type": "number", "description": "Pakuočių skaičius (vnt) arba kilogramai (kg)"},
                        },
                        "required": ["preke_id", "kiekis"],
                    },
                },
                "nerasta": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Prekės, kurių kataloge nėra, su kiekiu, pvz. 'avokadai (3 vnt.)'.",
                },
            },
            "required": ["prekes"],
        },
    ),
])

AGENTO_KONFIGURACIJA = types.GenerateContentConfig(
    system_instruction=SISTEMINE_INSTRUKCIJA,
    tools=[IRANKIU_APRASAI],
    # Funkcijas vykdome patys, kad matytume ir dokumentuotume kiekvieną žingsnį
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)
print("Įrankiai:", [f.name for f in IRANKIU_APRASAI.function_declarations])

### 5.3 Agento ciklas

Funkcija `paleisti_agenta()` įgyvendina agento ciklą ir kiekvieną žingsnį įrašo į **žurnalą** (`Zingsnis`):

1. **ĮVESTIS** — vartotojo tekstas perduodamas modeliui kartu su instrukcija ir įrankių aprašais.
2. **SPRENDIMAS** — modelis grąžina arba įrankio kvietimą (`function_call`), arba galutinį tekstą.
3. **ĮRANKIS** — jei kvietimas, kodas įvykdo funkciją ir rezultatą grąžina modeliui (`function_response`).
4. Ciklas kartojamas, kol modelis pateikia **GALUTINĮ ATSAKYMĄ** (arba pasiekiamas žingsnių limitas).

Galutinė išvestis — JSON objektas (atpažintas sąrašas, sumos, pigiausias tinklas, derinys) ir tekstinis atsakymas.

In [ ]:
@dataclass
class Zingsnis:
    nr: int
    tipas: str           # IVESTIS | IRANKIO_KVIETIMAS | IRANKIO_REZULTATAS | ATSAKYMAS
    pavadinimas: str
    turinys: object
    trukme_s: float = 0.0
    tokenai: int = 0


@dataclass
class AgentoRezultatas:
    uzklausa: str
    atsakymas: str = ""
    skaiciavimas: dict = field(default_factory=dict)
    zurnalas: list = field(default_factory=list)

    @property
    def isvestis(self) -> dict:
        # Galutinė struktūrizuota išvestis, to paties formato kaip etalonų y
        return {**self.skaiciavimas, "atsakymas": self.atsakymas}


SPALVOS = {"IVESTIS": "#2a78d6", "IRANKIO_KVIETIMAS": "#eb6834", "IRANKIO_REZULTATAS": "#1baf7a", "ATSAKYMAS": "#6250d6"}
ETIKETES = {"IVESTIS": "Įvestis", "IRANKIO_KVIETIMAS": "Modelis kviečia įrankį",
            "IRANKIO_REZULTATAS": "Įrankio rezultatas", "ATSAKYMAS": "Galutinis atsakymas"}


def rodyti_zingsni(z: Zingsnis):
    turinys = z.turinys if isinstance(z.turinys, str) else json.dumps(z.turinys, ensure_ascii=False, indent=2)
    if len(turinys) > 1500:
        turinys = turinys[:1500] + "\n… (sutrumpinta)"
    turinys = turinys.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    meta = f"{z.trukme_s:.2f} s" + (f" · {z.tokenai} tokenų" if z.tokenai else "")
    display(HTML(f'''
    <div style="border-left:4px solid {SPALVOS[z.tipas]};padding:6px 12px;margin:8px 0;background:rgba(127,127,127,.06)">
      <div style="font-weight:600">Žingsnis {z.nr}. {ETIKETES[z.tipas]}: {z.pavadinimas}
        <span style="font-weight:400;opacity:.65;float:right">{meta}</span></div>
      <pre style="white-space:pre-wrap;margin:6px 0 0 0;font-size:12px">{turinys}</pre>
    </div>'''))


def kviesti_modeli(contents, bandymai=4):
    # generate_content su pakartojimu, jei pasiekiamas užklausų limitas (429) ar laikina klaida (503)
    for bandymas in range(bandymai):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=AGENTO_KONFIGURACIJA)
        except Exception as e:
            if bandymas == bandymai - 1 or not any(k in str(e) for k in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")):
                raise
            laukti = 2 ** bandymas * 5
            print(f"  (API limitas, laukiama {laukti} s)")
            time.sleep(laukti)


def paleisti_agenta(uzklausa: str, rodyti: bool = True, max_zingsniu: int = 6) -> AgentoRezultatas:
    rez = AgentoRezultatas(uzklausa=uzklausa)

    def irasyti(z):
        rez.zurnalas.append(z)
        if rodyti:
            rodyti_zingsni(z)

    irasyti(Zingsnis(1, "IVESTIS", "vartotojo užklausa", uzklausa))
    contents = [types.Content(role="user", parts=[types.Part(text=uzklausa)])]

    for _ in range(max_zingsniu):
        t0 = time.time()
        atsakas = kviesti_modeli(contents)
        trukme = time.time() - t0
        tokenai = getattr(atsakas.usage_metadata, "total_token_count", 0) or 0

        if not atsakas.function_calls:                       # modelis baigė darbą
            rez.atsakymas = (atsakas.text or "").strip()
            irasyti(Zingsnis(len(rez.zurnalas) + 1, "ATSAKYMAS", MODEL, rez.atsakymas, trukme, tokenai))
            return rez

        contents.append(atsakas.candidates[0].content)       # modelio kvietimas lieka istorijoje
        atsakymo_dalys = []
        for fc in atsakas.function_calls:
            argumentai = dict(fc.args or {})
            irasyti(Zingsnis(len(rez.zurnalas) + 1, "IRANKIO_KVIETIMAS", f"{fc.name}()", argumentai, trukme, tokenai))
            t1 = time.time()
            try:
                rezultatas = IRANKIAI[fc.name](**argumentai)
            except Exception as e:                           # klaidą grąžiname modeliui, kad pataisytų
                rezultatas = {"klaida": f"{type(e).__name__}: {e}"}
            if fc.name == "apskaiciuoti_krepseli" and "klaida" not in rezultatas:
                rez.skaiciavimas = rezultatas
            irasyti(Zingsnis(len(rez.zurnalas) + 1, "IRANKIO_REZULTATAS", f"{fc.name}()", rezultatas, time.time() - t1))
            atsakymo_dalys.append(types.Part.from_function_response(name=fc.name, response={"result": rezultatas}))
            trukme, tokenai = 0.0, 0                          # modelio laikas priskirtas pirmam kvietimui
        contents.append(types.Content(role="user", parts=atsakymo_dalys))

    rez.atsakymas = "(Pasiektas žingsnių limitas, atsakymas nesugeneruotas.)"
    return rez

## 6. Demonstracija: nuo įvesties iki išvesties, žingsnis po žingsnio

Paleidžiame agentą su etalonine užklausa nr. 02 — joje yra kiekiai ir vienetai („1 kg“, „2 l“, „pusės kilogramo“), kuriuos modelis turi teisingai paversti katalogo vienetais.

Spalvos žurnale: **mėlyna** — įvestis, **oranžinė** — modelio sprendimas kviesti įrankį, **žalia** — įrankio rezultatas, **violetinė** — galutinis atsakymas.

In [ ]:
uzklausa = "Vakarienei reikės 1 kg vištienos filė, pakelio spagečių, 2 l pieno ir pusės kilogramo sūrio"  # @param {type:"string"}

demo = paleisti_agenta(uzklausa)

### Galutinė išvestis

Žemiau — tai, ką sistema grąžintų vartotojo sąsajai: žmogiškas atsakymas ir struktūrizuotas JSON.

In [ ]:
display(Markdown(f"> **Vartotojas:** {demo.uzklausa}\n>\n> **KrepšelisAI:** {demo.atsakymas}"))
print(json.dumps(demo.isvestis, ensure_ascii=False, indent=2))

### Agento žurnalas lentelėje

Ta pati eiga suvestinėje: kiek žingsnių, kiek laiko ir tokenų sunaudota.

In [ ]:
def zurnalo_lentele(rez: AgentoRezultatas) -> pd.DataFrame:
    eil = []
    for z in rez.zurnalas:
        t = z.turinys if isinstance(z.turinys, str) else json.dumps(z.turinys, ensure_ascii=False)
        eil.append({"Nr.": z.nr, "Etapas": ETIKETES[z.tipas], "Kas": z.pavadinimas,
                    "Trukmė, s": round(z.trukme_s, 2), "Tokenai": z.tokenai,
                    "Turinys (pradžia)": t[:90] + ("…" if len(t) > 90 else "")})
    return pd.DataFrame(eil).set_index("Nr.")

display(zurnalo_lentele(demo))
print(f"Iš viso: {sum(z.trukme_s for z in demo.zurnalas):.2f} s, {sum(z.tokenai for z in demo.zurnalas)} tokenų")

## 7. Rezultatų vizualizacija

1. **Krepšelio kaina kiekviename tinkle** su kortele ir be jos — iš karto matyti pigiausias tinklas.
2. **Iš ko susideda krepšelio kaina** — kiekvienos prekės indėlis į sumą (su kortele).
3. **Vienas tinklas ar du** — pigiausias vienas tinklas lyginamas su geriausiu 2 parduotuvių deriniu.

In [ ]:
TINKLO_SPALVA = {"Rimi": "#2a78d6", "IKI": "#eb6834", "Norfa": "#1baf7a"}
TEKSTAS, TEKSTAS2, TINKLELIS = "#0b0b0b", "#52514e", "#e4e3df"


def eur(x):
    return f"{x:.2f}".replace(".", ",") + " €"


def stilius(ax, pavadinimas):
    ax.set_title(pavadinimas, loc="left", fontsize=11.5, color=TEKSTAS, pad=10)
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(TEKSTAS2)
    ax.tick_params(colors=TEKSTAS2, length=0)
    ax.yaxis.grid(True, color=TINKLELIS, lw=0.8)
    ax.set_axisbelow(True)


def vizualizuoti(rez: AgentoRezultatas):
    s = rez.skaiciavimas
    if not s:
        print("Skaičiavimo rezultato nėra — nėra ką vizualizuoti.")
        return
    sumos, pig = s["sumos_eur"], s["pigiausias_tinklas"]["su_kortele"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), gridspec_kw={"width_ratios": [1.2, 1.3, 0.9]})

    # 1. Sumos su kortele ir be
    ax = axes[0]
    x = range(len(TINKLAI))
    w = 0.38
    for i, t in enumerate(TINKLAI):
        be, su = sumos[t]["be_korteles"], sumos[t]["su_kortele"]
        ax.bar(i - w / 2 - 0.01, be, w, color=TINKLO_SPALVA[t], alpha=0.35, edgecolor="white", linewidth=2)
        ax.bar(i + w / 2 + 0.01, su, w, color=TINKLO_SPALVA[t], edgecolor="white", linewidth=2)
        ax.text(i - w / 2 - 0.01, be * 0.97, eur(be).replace(" €", ""), ha="center", va="top", fontsize=8, color=TEKSTAS2)
        ax.text(i + w / 2 + 0.01, su, eur(su), ha="center", va="bottom", fontsize=9,
                color=TEKSTAS, fontweight="bold" if t == pig else "normal")
    ax.set_xticks(list(x), [t + ("\n(pigiausias)" if t == pig else "") for t in TINKLAI])
    ax.set_ylim(0, max(v["be_korteles"] for v in sumos.values()) * 1.18)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color="#8a8984", alpha=0.35, label="be kortelės"), Patch(color="#8a8984", label="su kortele")],
              frameon=False, loc="upper right", fontsize=8.5)
    stilius(ax, "Krepšelio kaina, €")

    # 2. Prekių indėlis (sukrautos juostos)
    ax = axes[1]
    prekes = s["atpazintas_sarasas"]
    pilkos = ["#3c3b38", "#6b6a66", "#9a9994", "#c3c2b7", "#dcdbd4", "#52514e", "#8a8984"]
    apacia = [0.0] * len(TINKLAI)
    for j, e in enumerate(prekes):
        reiksmes = [KAINOS[(e["preke_id"], t)]["su_kortele"] * e["kiekis"] for t in TINKLAI]
        ax.bar(TINKLAI, reiksmes, 0.55, bottom=apacia, color=pilkos[j % len(pilkos)], edgecolor="white", linewidth=2,
               label=f'{e["preke"]} × {e["kiekis"]:g} {e["vienetas"]}'.replace(".", ","))
        apacia = [a + r for a, r in zip(apacia, reiksmes)]
    for i, t in enumerate(TINKLAI):
        ax.text(i, apacia[i], eur(sumos[t]["su_kortele"]), ha="center", va="bottom", fontsize=9, color=TEKSTAS)
    ax.set_ylim(0, max(apacia) * 1.15)
    h, l = ax.get_legend_handles_labels()
    ax.legend(h[::-1], l[::-1], frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    stilius(ax, "Iš ko susideda kaina (su kortele), €")

    # 3. Vienas tinklas ar derinys
    ax = axes[2]
    d = s["geriausias_2_parduotuviu_derinys"]
    vardai = [f"{pig}", " + ".join(d["tinklai"])]
    reiksmes = [sumos[pig]["su_kortele"], d["suma"]]
    spalvos = [TINKLO_SPALVA[pig], "#6250d6"]
    ax.bar(vardai, reiksmes, 0.5, color=spalvos, edgecolor="white", linewidth=2)
    for i, v in enumerate(reiksmes):
        ax.text(i, v, eur(v), ha="center", va="bottom", fontsize=9, color=TEKSTAS)
    ax.set_ylim(0, max(reiksmes) * 1.2)
    sprendimas = "verta" if d["verta"] else "neverta (< 1 €)"
    ax.set_xlabel(f"Derinys sutaupo {eur(d['sutaupoma_lyginant_su_pigiausiu'])} — {sprendimas}", color=TEKSTAS2, fontsize=9)
    stilius(ax, "Vienas tinklas ar du, €")

    fig.suptitle(f"„{rez.uzklausa}“", x=0.01, ha="left", fontsize=10, color=TEKSTAS2, y=1.02)
    plt.tight_layout()
    plt.show()


vizualizuoti(demo)

In [ ]:
# Agento eigos laiko juosta: kiek laiko užėmė kiekvienas žingsnis
def laiko_juosta(rez: AgentoRezultatas):
    fig, ax = plt.subplots(figsize=(12, 0.6 + 0.45 * len(rez.zurnalas)))
    pradzia = 0.0
    for i, z in enumerate(rez.zurnalas):
        plotis = max(z.trukme_s, 0.02)
        ax.barh(i, plotis, left=pradzia, color=SPALVOS[z.tipas], height=0.6, edgecolor="white", linewidth=2)
        ax.text(pradzia + plotis + 0.03, i, f"{ETIKETES[z.tipas]}: {z.pavadinimas}  ({z.trukme_s:.2f} s)",
                va="center", fontsize=9, color=TEKSTAS)
        pradzia += z.trukme_s
    ax.set_yticks(range(len(rez.zurnalas)), [f"Žingsnis {z.nr}" for z in rez.zurnalas])
    ax.invert_yaxis()
    ax.set_xlim(0, pradzia * 1.6 + 0.5)
    ax.set_xlabel("laikas nuo užklausos pradžios, s", color=TEKSTAS2)
    for sp in ["top", "right", "left"]:
        ax.spines[sp].set_visible(False)
    ax.tick_params(colors=TEKSTAS2, length=0)
    ax.set_title("Agento eiga laike", loc="left", fontsize=11.5, color=TEKSTAS)
    plt.tight_layout(); plt.show()

laiko_juosta(demo)

## 8. Išbandykite patys

Įrašykite savo pirkinių sąrašą ir paleiskite langelį. Katalogo prekės: pienas, ruginė duona, kiaušiniai, sviestas, vištienos filė, spagečiai, fermentinis sūris, bananai, obuoliai, malta kava. Galite įrašyti ir prekę, kurios kataloge nėra — agentas turėtų ją pažymėti kaip nerastą.

In [ ]:
mano_uzklausa = "Savaitgaliui: 3 pakeliai spagečių, kilogramas obuolių, 2 sviestai ir pakelis kavos, dar norėčiau ananaso"  # @param {type:"string"}

mano = paleisti_agenta(mano_uzklausa)
display(Markdown(f"> **KrepšelisAI:** {mano.atsakymas}"))
vizualizuoti(mano)

## 9. Vertinimas su etaloninėmis (X, y) poromis

Agentą paleidžiame su visomis 4 etaloninėmis užklausomis **X** ir jo išvestį lyginame su teisingu atsakymu **y**:

| Kriterijus | Kaip tikrinama |
|---|---|
| Atpažintas sąrašas | Sutampa `preke_id` ir kiekiai |
| Nerastos prekės | Sutampa nerastų prekių skaičius |
| Sumos | Visų 3 tinklų sumos su kortele ir be sutampa ±0,01 € tikslumu |
| Pigiausias tinklas | Sutampa su kortele ir be kortelės |
| Derinys | Sutampa tinklų pora, suma ir sprendimas „verta“ |

Teksto formuluotė gali skirtis — tikriname faktus, ne žodžius.

In [ ]:
def palyginti(rez: AgentoRezultatas, y: dict) -> dict:
    s = rez.skaiciavimas or {}
    sar = lambda lst: sorted((e["preke_id"], round(float(e["kiekis"]), 3)) for e in lst)
    d_s, d_y = s.get("geriausias_2_parduotuviu_derinys", {}), y["geriausias_2_parduotuviu_derinys"]
    return {
        "Atpažintas sąrašas": bool(s) and sar(s["atpazintas_sarasas"]) == sar(y["atpazintas_sarasas"]),
        "Nerastos prekės": bool(s) and len(s["nerasta"]) == len(y["nerasta"]),
        "Sumos": bool(s) and artimi(s["sumos_eur"], y["sumos_eur"]),
        "Pigiausias tinklas": bool(s) and s["pigiausias_tinklas"] == y["pigiausias_tinklas"],
        "Derinys": bool(s) and sorted(d_s.get("tinklai", [])) == sorted(d_y["tinklai"])
                   and artimi(d_s.get("suma"), d_y["suma"]) and d_s.get("verta") == d_y["verta"],
    }


vertinimas, rezultatai = [], []
for i, et in enumerate(etalonai, 1):
    rez = paleisti_agenta(et["X"], rodyti=False)
    rezultatai.append(rez)
    kriterijai = palyginti(rez, et["y"])
    vertinimas.append({"Nr.": f"{i:02d}", "Užklausa (X)": et["X"], **kriterijai,
                       "Žingsniai": len(rez.zurnalas),
                       "Laikas, s": round(sum(z.trukme_s for z in rez.zurnalas), 2),
                       "Tokenai": sum(z.tokenai for z in rez.zurnalas)})
    print(f"{i:02d}: {'✓' if all(kriterijai.values()) else '✗'}  {rez.atsakymas}")
    time.sleep(1)   # švelniai dėl nemokamo plano užklausų limito

vertinimo_df = pd.DataFrame(vertinimas).set_index("Nr.")
KRITERIJAI = ["Atpažintas sąrašas", "Nerastos prekės", "Sumos", "Pigiausias tinklas", "Derinys"]
display(vertinimo_df.style
        .format({k: lambda v: "✓" if v else "✗" for k in KRITERIJAI})
        .map(lambda v: "background-color:#e3f5ee" if v is True else ("background-color:#fbe4e4" if v is False else ""),
             subset=KRITERIJAI))

In [ ]:
# Tikslumas pagal kriterijų
tikslumas = vertinimo_df[KRITERIJAI].mean() * 100
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.barh(KRITERIJAI, tikslumas.values, color="#2a78d6", height=0.55, edgecolor="white", linewidth=2)
for i, v in enumerate(tikslumas.values):
    ax.text(v + 1, i, f"{v:.0f} %", va="center", fontsize=9.5, color=TEKSTAS)
ax.set_xlim(0, 112); ax.invert_yaxis()
for sp in ["top", "right", "left"]:
    ax.spines[sp].set_visible(False)
ax.tick_params(colors=TEKSTAS2, length=0)
ax.xaxis.grid(True, color=TINKLELIS); ax.set_axisbelow(True)
ax.set_title(f"Agento tikslumas su {len(etalonai)} etaloninėmis poromis, %", loc="left", fontsize=11.5, color=TEKSTAS)
plt.tight_layout(); plt.show()

visiskai = vertinimo_df[KRITERIJAI].all(axis=1).sum()
print(f"Visiškai teisingi atsakymai: {visiskai} iš {len(vertinimo_df)}")
print(f"Vidutinė trukmė: {vertinimo_df['Laikas, s'].mean():.2f} s, vidutiniškai {vertinimo_df['Tokenai'].mean():.0f} tokenų užklausai")

In [ ]:
# Visų 4 etaloninių užklausų rezultatai grafiškai
for rez in rezultatai:
    vizualizuoti(rez)

## 10. Išvados ir apribojimai

### Kas pademonstruota
- **Veikiantis DI agentas** su Gemini API: modelis pats planuoja veiksmus ir kviečia du įrankius (`gauti_kataloga`, `apskaiciuoti_krepseli`).
- **Visa eiga dokumentuota:** kiekvienas žingsnis nuo vartotojo įvesties iki galutinio atsakymo įrašomas į žurnalą su trukme ir tokenų skaičiumi.
- **Atsakomybės padalijimas:** DI supranta kalbą ir formuluoja atsakymą, o skaičiavimus atlieka deterministinis kodas. Todėl sumos visada tikslios, o klaidų galima tikėtis tik atpažinimo žingsnyje.
- **Vertinimas** su etaloninėmis (X, y) poromis leidžia objektyviai matuoti kokybę ir aptikti regresijas keičiant modelį ar instrukciją.

### Apribojimai
- Katalogas mažas (10 prekių) ir kainos pavyzdinės; tikroje sistemoje reikėtų kasdien atnaujinamų kainų iš scraperių ir duomenų bazės.
- LLM atsakymai nėra visiškai deterministiniai: ta pati užklausa retais atvejais gali būti atpažinta kitaip (pvz., „pakelis kavos“ = 1 vnt.).
- Nemokamas Gemini API planas turi užklausų per minutę limitą, todėl vertinimas vykdomas nuosekliai.

### Tolimesni žingsniai
1. Prijungti tikrų kainų duomenų bazę (PostgreSQL) ir scraperius Rimi, IKI, Norfa svetainėms.
2. Pridėti prekių paieškos įrankį su vektoriniais įterpiniais (*embeddings*), kad agentas rastų prekes ir didelėje kataloge.
3. Išplėsti etaloninių porų rinkinį iki 50+ užklausų ir vertinimą paleisti automatiškai po kiekvieno pakeitimo.
4. Perkelti agentą į Node.js serverį ir sujungti su React pokalbių sąsaja pagal projekto architektūrą.